In [1]:
import os

PROJECT_ROOT = "/Users/zanderholleran/Desktop/python_projects/mesa_lcc_model"
os.chdir(PROJECT_ROOT)

%pwd



# Enable auto-reloading of custom modules
%load_ext autoreload
%autoreload 2

In [2]:
from season.persons import SeasonPerson
from season.configs import ScheduleSpecs, SeasonConfig, DayParams, make_season_config, PopulationParams
from season.season_orchestrator import SeasonOrchestrator

import numpy as np
import pandas as pd
from scipy.stats import norm, lognorm, skewnorm, truncnorm, uniform

# Mesa model components
import mesa
from mesa import model
from mesa import agent
from traffic.model.traffic_model import TrafficModel
from traffic.agents.vehicle_agent import VehicleAgent
from traffic.agents.road_segment_agent import RoadSegmentAgent

import traffic.utils.unit_conversion_utils as uc
import traffic.utils.analysis_utils as au
import traffic.utils.animation_utils as anim

# Visualization and display (Jupyter-specific)
from IPython.display import display, HTML
pd.set_option("display.max_rows", None)  

# Define the config

In [14]:
pop_params = PopulationParams(
    population_size=1000,
    prior_car=22.0,
    prior_bus=40.0,

)


config = make_season_config(
    season_id='one_day',
    run_description='One day test run',
    seed=124,
    n_days=1,
    max_steps=4000,
    max_persons=1000,
    collect_every_n=10,
    batch_run=False, 

    start_hr=8,
    bus_capacity=30,
    road_path='data/roads/hw210_sl_and_curvs.parquet',
    ecs_path='data/vehicle_counts/expected_counts_seconds.csv',
    toll_mechanism='static',
    toll_params={'car': 8.0, 'bus': 0.0},
   
    traffic_percentile_schedule=ScheduleSpecs(mode='static', value=95),  # static bus interval of 15 minutes,
    bus_interval_schedule=ScheduleSpecs(mode='static', value=30),
    crashes_schedule=ScheduleSpecs(mode='static', value=4), 
    population_params=pop_params,
    canyon_closures_schedule=None,
)

# Run single day

In [15]:
# Example usage of SeasonOrchestrator with example_config
orchestrator = SeasonOrchestrator(season_config=config)
tm = orchestrator.run_day()

Toll mechanism: static with params {'car': 8.0, 'bus': 0.0}


Simulating:   9%|▉         | 379/4000 [00:00<00:11, 327.25step/s]

crash at step 332


Simulating: 100%|██████████| 4000/4000 [00:32<00:00, 123.07step/s]

Reached max step count (4000). Stopping model.


In [16]:
orchestrator.season_persons[0]

SeasonPerson(person_id=0, season_id='one_day', value_of_time=0.5499848644664439, experience_weight_car=1.0, experience_weight_bus=1.5135282511936452, prior_car=22.568333333333335, prior_bus=40.0, time_decay_rate=0.1, prior_weight=1.0, uncertainty_multiplier=1.0, travel_propensity=1.0, history=[{'day_index': 0, 'mode': 'car', 'realized_tt': 33.36666666666667, 'toll_paid': 8.0, 'cumtime_lost_sec': 941.0399163980937}], realized_costs=[], expected_tt_car=22.0, expected_tt_bus=40.0, travel_time_uncertainty_car=1.0, travel_time_uncertainty_bus=1.0)

# Analysis

In [ ]:
finished_agents = au.finished_agents_summary_df(tm, plots=True)
finished_agents.head()

In [ ]:
# process the finished_agents data 
vehicles_full = au.vehicle_agent_data_time_series(tm, plots=True)
model_ts = tm.datacollector.get_model_vars_dataframe()


au.plot_speed_delta(vehicles_full, model_ts)

# Annimations

In [ ]:
# run the animation
# looking at one car
issue_car_id =  604
issue_step = 0

In [ ]:
anim.animate_traffic(vehicles_full, road_gdf, interval=100, step_skip=2, watch=None, zoom=20)

In [ ]:
anim.animate_traffic_with_speed_delta_highlight(vehicles_full, road_gdf, model_ts, interval=100, step_skip=3, watch=None, zoom=20)

In [ ]:
anim.animate_relative_distance(vehicle_df=vehicles_full, agent_id=700, distance_behind=100, color_by='status')

In [18]:
import pandas as pd

# columns you want in the final CSV
cols = [
    "seed",
    "traffic_percentile",
    "car_toll",
    "avg_cumtime_lost_sec",
    "bus_prior",
    "car_prior",
]

results_df = pd.DataFrame(columns=cols)

orchestrator = SeasonOrchestrator(season_config=config)

# single run:
results_df = orchestrator.run_day(results_df)


# later: save to CSV / parquet
results_df.to_csv("single_day_experiment_results.csv", index=False)


TypeError: SeasonOrchestrator.run_day() takes 1 positional argument but 2 were given

In [ ]:
# or in a loop over configs / seeds:
for cfg in configs_to_try:
    orchestrator = SeasonOrchestrator(config=cfg)
    tm, results_df = orchestrator.run_day(results_df)
